## Import libraries

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import config
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import ParameterSampler
from xgboost import XGBClassifier

## Functions

In [2]:
def create_model(params=None):
    if params is None:
        params = best_xgb_params  # Defined after randomized search

    model_params = {
        "reg_alpha": 1.0,
        "tree_method": "hist",
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "n_jobs": -1,
        "random_state": config.RANDOM_STATE,
        **params,
    }

    return XGBClassifier(**model_params)

def X(data, columns):
    return data.loc[:, columns].replace([np.inf, -np.inf], np.nan)

def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan

def safe_ap(y, p):
    return float(average_precision_score(y, p)) if np.unique(y).size == 2 else np.nan

def direction_metrics(data, threshold):
    y = data[TARGET].to_numpy(dtype=int)
    p = data["Probability"].to_numpy(dtype=float)
    predicted_positive = p >= threshold
    selected = int(predicted_positive.sum())
    true_positive = int(np.sum(predicted_positive & (y == 1)))
    false_positive = int(np.sum(predicted_positive & (y == 0)))
    true_negative = int(np.sum(~predicted_positive & (y == 0)))
    false_negative = int(np.sum(~predicted_positive & (y == 1)))
    precision = true_positive / selected if selected else np.nan
    base = float(y.mean())
    return {
        "Observations": len(y),
        "Always_Positive_Precision": base,
        "Precision": precision,
        "Precision_Lift_pp": 100.0 * (precision - base) if selected else np.nan,
        "Recall": true_positive / (true_positive + false_negative) if np.any(y == 1) else np.nan,
        "Specificity": true_negative / (true_negative + false_positive) if np.any(y == 0) else np.nan,
        "Balanced_Accuracy": float(balanced_accuracy_score(y, predicted_positive)),
        "Positive_Prediction_Rate": selected / len(y),
        "Predicted_Positive_Count": selected,
        "ROC_AUC": safe_auc(y, p),
        "Average_Precision": safe_ap(y, p),
    }

def screening_summary(part):
    n_selected = int(part["Selected_Assets"].sum())
    n_correct = int(part["Correct_Positives"].sum())
    total_dates = len(part)
    n_active = int((part["Selected_Assets"] > 0).sum())
    precision = n_correct / n_selected if n_selected else np.nan
    matched_baseline = (
        float((part["Selected_Assets"] * part["Same_Date_Baseline"]).sum() / n_selected)
        if n_selected else np.nan
    )
    return {
        "Dates": total_dates,
        "Selected_Asset_Dates": n_selected,
        "Precision": precision,
        "Matched_Date_Baseline": matched_baseline,
        "Precision_Lift_pp": 100 * (precision - matched_baseline) if n_selected else np.nan,
        "Mean_Assets": n_selected / total_dates,
        "Min_Assets": int(part["Selected_Assets"].min()),
        "Max_Assets": int(part["Selected_Assets"].max()),
        "Cash_Only_Dates": total_dates - n_active,
    }

## Load features and define feature sets

In [3]:
# Output folder and model-selection mode
OUTPUT = config.OUTPUT_DIR / "classification_results/xgboost"
OUTPUT.mkdir(parents=True, exist_ok=True)

# Screen every asset using only these inclusive probability cutoffs
CLASSIFICATION_THRESHOLDS = config.CLASSIFICATION_THRESHOLDS
# Minimum mean selected assets per validation rebalance is set in config.py
MIN_AVERAGE_SELECTED_ASSETS = config.MIN_AVERAGE_SELECTED_ASSETS

if not isinstance(config.HYPERPARAMETER_TUNING, bool):
    raise TypeError("HYPERPARAMETER_TUNING must be True or False.")

ALGORITHMS = ["XGBoost"]
selected_config_path = OUTPUT / "selected_config.json"
loaded_config = None
best_xgb_params = None

if not config.HYPERPARAMETER_TUNING:
    if not selected_config_path.is_file():
        raise FileNotFoundError(
            f"Hyperparameter tuning is disabled, but {selected_config_path} does not exist. "
            "Run once with HYPERPARAMETER_TUNING=True to save a validated configuration."
        )
    with selected_config_path.open("r", encoding="utf-8") as file:
        loaded_config = json.load(file)
    if loaded_config.get("algorithm") != "XGBoost":
        raise ValueError("Saved selected_config.json is not an XGBoost configuration.")
    if not isinstance(loaded_config.get("model_params"), dict):
        raise ValueError("Saved configuration is missing model_params.")
    if not isinstance(loaded_config.get("screening"), dict):
        raise ValueError("Saved configuration is missing screening.")
    best_xgb_params = dict(loaded_config["model_params"])
    print(f"Hyperparameter tuning skipped. Loaded saved model parameters (screening reselected on validation) from {selected_config_path}")
else:
    print("Hyperparameter tuning enabled: randomized search will run on validation data.")

# Set features folder
FEATURES_DATA_DIR = config.INPUT_DATA_DIR.parent / "features_data"

# Load asset classes from the investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

# Read features files
df = pd.read_csv(FEATURES_DATA_DIR / "combined_features.csv", parse_dates=["Date"])
regimes = pd.read_csv(FEATURES_DATA_DIR / "market_regimes.csv", parse_dates=["Date"])

# Merge with confirmed regime label
df = df.merge(
    regimes[["Date", "Market_Regime"]],
    on="Date",
    how="left",
    validate="many_to_one"
)

df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

# Identify dates with missing regime labels
missing_dates = (df.loc[df["Market_Regime"].isna(), "Date"].drop_duplicates().sort_values())
missing_count = df["Market_Regime"].isna().sum()

# Forward fill using the last available confirmed regime
df["Market_Regime"] = df["Market_Regime"].ffill()

# Print dates and assigned regime labels
filled_dates = (
    df.loc[df["Date"].isin(missing_dates), ["Date", "Market_Regime"]]
    .drop_duplicates()
    .sort_values("Date")
)

print(f"Forward-filled regime dates: {len(missing_dates)}")

# Final checks
print("\nDataset shape:", df.shape)
print("Missing regime labels:", df["Market_Regime"].isna().sum())
print("Duplicate asset-dates:", df.duplicated(["Date", "Ticker"]).sum())
print(df["Target_60d"].value_counts(normalize=True))

asset_classes = universe[["Ticker", "Asset Class"]].drop_duplicates(subset="Ticker")

# Merge asset classes into the modeling dataset
df = df.merge(
    asset_classes,
    on="Ticker",
    how="left",
    validate="many_to_one"
)

# Encode asset classes
asset_class_dummies = pd.get_dummies(df["Asset Class"], prefix="AssetClass", dtype=int)
df = pd.concat([df, asset_class_dummies], axis=1)
asset_class_features = asset_class_dummies.columns.tolist()

print("\nAsset-class features:")
print(asset_class_features)

# One-hot encode ticker identity
ticker_dummies = pd.get_dummies(df["Ticker"], prefix="Ticker", dtype=int)
df = pd.concat([df, ticker_dummies], axis=1)
ticker_features = ticker_dummies.columns.tolist()

# Define target label
TARGET = "Target_60d"

# Columns exclude from features
EXCLUDE = ["Date", "Ticker", "Forward_Return_60d", "Target_60d", "Market_Regime", "Asset Class"]

# Define feature set
base_features = [c for c in df.columns if c not in EXCLUDE and not c.startswith("Regime_") and not c.startswith("Ticker_")]

# Use one-hot encoding for confirmed regimes
regime_dummies = pd.get_dummies(df["Market_Regime"], prefix="Regime", dtype=int)
df = pd.concat([df, regime_dummies], axis=1)
regime_features = regime_dummies.columns.tolist()

# Regime specifications
features_without_regime = base_features
features_with_regime = base_features + regime_features

# Ticker-level specifications
features_with_tickers = base_features + ticker_features
features_with_tickers_regime = (base_features + ticker_features + regime_features)

SPECIFICATIONS = {
    "Without Regimes": features_with_tickers,
    "With Regimes": features_with_tickers_regime,
}

for name, features in SPECIFICATIONS.items():
    assert len(features) == len(set(features)), name
    assert not set(["Date", "Ticker", TARGET, "Forward_Return_60d"]).intersection(features)

Hyperparameter tuning enabled: randomized search will run on validation data.
Forward-filled regime dates: 0

Dataset shape: (217100, 256)
Missing regime labels: 0
Duplicate asset-dates: 0
Target_60d
1    0.613418
0    0.386582
Name: proportion, dtype: float64

Asset-class features:
['AssetClass_Commodity', 'AssetClass_Crypto', 'AssetClass_Equity', 'AssetClass_Fixed Income', 'AssetClass_Foreign Exchange', 'AssetClass_Volatility']


## Walk-forward validation

In [4]:
# Validation period and rebalance frequency
VALIDATION_START = pd.Timestamp(config.VALIDATION_START)
TEST_START = pd.Timestamp(config.TEST_START)
REBALANCE_STEP = config.REBALANCE_FREQUENCY

# Calculate when each asset's 60-session target becomes known
df = df.sort_values(["Ticker", "Date"], kind="stable").reset_index(drop=True).copy()
df["_Target_End_Date"] = (
    df.groupby("Ticker", sort=False)["Date"].shift(-config.FORWARD_HORIZON)
)

valid_target = (
    df[TARGET].isin([0, 1])
    & df["Forward_Return_60d"].notna()
    & df["_Target_End_Date"].notna()
)

# Use the dates already available in the merged dataframe
trading_dates = pd.DatetimeIndex(sorted(df["Date"].dropna().unique()))

# Anchor the schedule to the first 2022 rebalance date
test_anchor = trading_dates[trading_dates >= TEST_START][0]
anchor_pos = trading_dates.get_loc(test_anchor)

# Generate 60-session rebalance dates backwards from 2022
positions = np.arange(anchor_pos - REBALANCE_STEP, -1, -REBALANCE_STEP)

validation_dates = trading_dates[positions]
validation_dates = validation_dates[
    (validation_dates >= VALIDATION_START)
    & (validation_dates < TEST_START)
].sort_values()

# Prepare expanding-window training and validation datasets
walk_forward_splits = []
validation_summary = []
skipped_dates = []

for rebalance_date in validation_dates:

    train = df.loc[
        (df["Date"] < rebalance_date)
        & (df["_Target_End_Date"] < rebalance_date)
        & valid_target
    ].copy()

    validation = df.loc[df["Date"] == rebalance_date].copy()

    # Require complete, known outcomes for the rebalance date
    known = (valid_target.loc[validation.index] & (validation["_Target_End_Date"] < TEST_START))

    if validation.empty or not known.all():
        skipped_dates.append(rebalance_date)
        continue

    assert not train.empty, rebalance_date
    assert train[TARGET].nunique() == 2, rebalance_date
    assert train["_Target_End_Date"].max() < rebalance_date
    assert validation["Date"].nunique() == 1
    assert validation["_Target_End_Date"].max() < TEST_START

    walk_forward_splits.append((rebalance_date, train, validation))

    validation_summary.append({
        "Rebalance_Date": rebalance_date,
        "Train_Start": train["Date"].min(),
        "Train_End": train["Date"].max(),
        "Train_Obs": len(train),
        "Val_Obs": len(validation),
        "Val_Baseline": validation[TARGET].mean(),
    })

# Ensure that no asset is evaluated twice on the same date
validation_assets = pd.concat(
    [v[["Date", "Ticker"]] for _, _, v in walk_forward_splits],
    ignore_index=True
)

assert not validation_assets.duplicated(["Date", "Ticker"]).any()

print("Scheduled validation dates:", len(validation_dates))
print("Completed validation dates:", len(walk_forward_splits))
print("Excluded immature/incomplete dates:", len(skipped_dates))

display(pd.DataFrame(validation_summary))

Scheduled validation dates: 25
Completed validation dates: 24
Excluded immature/incomplete dates: 1


,Rebalance_Date,Train_Start,Train_End,Train_Obs,Val_Obs,Val_Baseline
0,2016-01-20,2008-01-02,2015-10-21,86587,48,0.958333
1,2016-04-15,2008-01-02,2016-01-19,89467,48,0.833333
2,2016-07-12,2008-01-02,2016-04-14,92347,48,0.541667
3,2016-10-05,2008-01-02,2016-07-11,95227,48,0.333333
4,2016-12-30,2008-01-02,2016-10-04,98107,48,0.895833
5,2017-03-29,2008-01-02,2016-12-29,100987,48,0.833333
6,2017-06-23,2008-01-02,2017-03-28,103867,48,0.875000
7,2017-09-19,2008-01-02,2017-06-22,106747,48,0.666667
8,2017-12-13,2008-01-02,2017-09-18,109627,48,0.625000
9,2018-03-13,2008-01-02,2017-12-12,112507,48,0.541667


## Hyperparameter search space

In [5]:
# Run randomized search when tuning is enabled.
if config.HYPERPARAMETER_TUNING:
    TUNING_SPECIFICATION = "With Regimes"
    N_ITER = 30

    XGB_PARAM_SPACE = {
        "max_depth": [1, 2, 3, 4],
        "n_estimators": [75, 100, 150, 250],
        "learning_rate": [0.02, 0.05, 0.10],
        "min_child_weight": [20, 50, 100],
        "reg_lambda": [1, 10, 20, 50],
        "subsample": [0.70, 0.85, 1.00],
        "colsample_bytree": [0.70, 0.85, 1.00],
    }

    parameter_sets = list(ParameterSampler(
        XGB_PARAM_SPACE,
        n_iter=N_ITER,
        random_state=config.RANDOM_STATE,
    ))


    print(f"XGBoost parameter configurations: {len(parameter_sets)}")
    print(f"Walk-forward validation dates: {len(walk_forward_splits)}")
    print(f"Search model fits: {len(parameter_sets) * len(walk_forward_splits)}")

XGBoost parameter configurations: 30
Walk-forward validation dates: 24
Search model fits: 720


## Hyperparameter randomized search

In [6]:
if config.HYPERPARAMETER_TUNING:
    tuning_features = SPECIFICATIONS[TUNING_SPECIFICATION]
    search_daily_rows = []

    with tqdm(
        total=len(parameter_sets) * len(walk_forward_splits),
        desc="XGBoost random search",
        unit="fit",
    ) as progress:
        for trial, params in enumerate(parameter_sets, start=1):
            for rebalance_date, train, validation in walk_forward_splits:
                y_train = train[TARGET].astype(int).to_numpy()
                y_val = validation[TARGET].astype(int).to_numpy()

                model = create_model(params=params)
                model.fit(X(train, tuning_features), y_train)

                p_train = model.predict_proba(X(train, tuning_features))[:, 1]
                p_val = model.predict_proba(X(validation, tuning_features))[:, 1]

                train_auc = safe_auc(y_train, p_train)
                val_auc = safe_auc(y_val, p_val)

                search_daily_rows.append({
                    "Trial": trial,
                    "Rebalance_Date": rebalance_date,
                    "Train_Observations": len(y_train),
                    "Validation_Observations": len(y_val),
                    "Train_ROC_AUC": train_auc,
                    "Val_ROC_AUC": val_auc,
                    "Val_Average_Precision": safe_ap(y_val, p_val),
                    "Val_Log_Loss": float(log_loss(y_val, p_val, labels=[0, 1])),
                    "AUC_Gap": train_auc - val_auc,
                })

                progress.set_postfix(trial=trial, date=rebalance_date.strftime("%Y-%m-%d"))
                progress.update(1)

    search_by_rebalance = pd.DataFrame(search_daily_rows)

    parameter_table = pd.DataFrame([
        {"Trial": trial, **params, "Parameters_JSON": json.dumps(params, sort_keys=True)}
        for trial, params in enumerate(parameter_sets, start=1)
    ])

    hyperparameter_search_results = (
        search_by_rebalance.groupby("Trial", as_index=False)
        .agg(
            Validation_Dates=("Rebalance_Date", "nunique"),
            Val_ROC_AUC_Mean=("Val_ROC_AUC", "mean"),
            Val_ROC_AUC_SD=("Val_ROC_AUC", "std"),
            Valid_AUC_Dates=("Val_ROC_AUC", "count"),
            Val_AP_Mean=("Val_Average_Precision", "mean"),
            Val_Log_Loss_Mean=("Val_Log_Loss", "mean"),
            Train_ROC_AUC_Mean=("Train_ROC_AUC", "mean"),
            AUC_Gap_Mean=("AUC_Gap", "mean"),
        )
        .merge(parameter_table, on="Trial", validate="one_to_one")
        .sort_values(
            ["Val_ROC_AUC_Mean", "Val_AP_Mean", "AUC_Gap_Mean"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    assert (hyperparameter_search_results["Validation_Dates"] == len(walk_forward_splits)).all()
    assert hyperparameter_search_results["Valid_AUC_Dates"].nunique() == 1
    assert hyperparameter_search_results["Val_ROC_AUC_Mean"].notna().all()

    # Highest mean validation ROC AUC wins. AP and AUC gap break exact ties.
    best_trial = int(hyperparameter_search_results.loc[0, "Trial"])
    best_xgb_params = dict(parameter_sets[best_trial - 1])
    hyperparameter_search_results["Selected"] = (
        hyperparameter_search_results["Trial"] == best_trial
    )

    best_hyperparameters = pd.DataFrame([{
        "Algorithm": "XGBoost",
        "Tuning_Specification": TUNING_SPECIFICATION,
        "Trial": best_trial,
        "Validation_Dates": len(walk_forward_splits),
        "Selection_Metric": "Mean within-date validation ROC AUC",
        "Val_ROC_AUC_Mean": hyperparameter_search_results.loc[0, "Val_ROC_AUC_Mean"],
        "Val_AP_Mean": hyperparameter_search_results.loc[0, "Val_AP_Mean"],
        **best_xgb_params,
    }])

    print("Selected randomized-search configuration:")
    print(json.dumps(best_xgb_params, indent=2, sort_keys=True))
    display(hyperparameter_search_results[[
        "Trial", "Selected", "Val_ROC_AUC_Mean", "Val_ROC_AUC_SD",
        "Val_AP_Mean", "Val_Log_Loss_Mean", "Train_ROC_AUC_Mean",
        "AUC_Gap_Mean", *XGB_PARAM_SPACE.keys(),
    ]].round(4))

XGBoost random search:   0%|          | 0/720 [00:00<?, ?fit/s]

Selected randomized-search configuration:
{
  "colsample_bytree": 0.7,
  "learning_rate": 0.05,
  "max_depth": 2,
  "min_child_weight": 100,
  "n_estimators": 250,
  "reg_lambda": 20,
  "subsample": 1.0
}


,Trial,Selected,Val_ROC_AUC_Mean,Val_ROC_AUC_SD,Val_AP_Mean,Val_Log_Loss_Mean,Train_ROC_AUC_Mean,AUC_Gap_Mean,max_depth,n_estimators,learning_rate,min_child_weight,reg_lambda,subsample,colsample_bytree
0,1,True,0.6866,0.2009,0.7500,0.6463,0.8295,0.1429,2,250,0.05,100,20,1.00,0.70
1,19,False,0.6760,0.2117,0.7486,0.6684,0.8548,0.1788,2,250,0.10,50,20,0.85,1.00
2,3,False,0.6706,0.2000,0.7366,0.6500,0.8239,0.1534,2,100,0.10,50,20,1.00,0.85
3,28,False,0.6687,0.1969,0.7401,0.6606,0.8002,0.1315,2,250,0.02,20,50,0.70,0.70
4,10,False,0.6641,0.2146,0.7465,0.6610,0.8820,0.2179,4,75,0.10,100,10,1.00,0.70
5,29,False,0.6639,0.2053,0.7420,0.6783,0.8650,0.2011,3,250,0.05,100,50,0.85,0.85
6,26,False,0.6579,0.2053,0.7300,0.6583,0.8353,0.1774,3,100,0.05,100,10,0.70,0.70
7,7,False,0.6564,0.2015,0.7429,0.6652,0.8537,0.1972,3,75,0.10,20,10,0.70,0.85
8,22,False,0.6559,0.2095,0.7385,0.6589,0.8869,0.2310,4,150,0.05,50,1,1.00,0.70
9,5,False,0.6552,0.2173,0.7289,0.6731,0.8264,0.1712,3,75,0.05,100,50,0.85,1.00


## Retrain tuned XGBoost and generate validation probabilities

In [7]:
prediction_parts = []
fit_rows = []

for rebalance_date, train, validation in tqdm(
    walk_forward_splits,
    desc="Tuned XGBoost walk-forward",
    unit="rebalance",
):
    y_train = train[TARGET].astype(int).to_numpy()
    y_val = validation[TARGET].astype(int).to_numpy()

    for algorithm in ALGORITHMS:
        for specification, features in SPECIFICATIONS.items():
            model = create_model()
            model.fit(X(train, features), y_train)

            train_p = model.predict_proba(X(train, features))[:, 1]
            val_p = model.predict_proba(X(validation, features))[:, 1]

            for stage, y, p in [
                ("Train", y_train, train_p),
                ("Validation", y_val, val_p),
            ]:
                fit_rows.append({
                    "Rebalance_Date": rebalance_date,
                    "Algorithm": algorithm,
                    "Specification": specification,
                    "Stage": stage,
                    "Observations": len(y),
                    "Baseline_Precision": float(y.mean()),
                    "ROC_AUC": safe_auc(y, p),
                    "Average_Precision": safe_ap(y, p),
                    "Log_Loss": float(log_loss(y, p, labels=[0, 1])),
                    **{
                        f"Precision_{cutoff:.2f}": (
                            float(y[p >= cutoff].mean())
                            if np.any(p >= cutoff) else np.nan
                        )
                        for cutoff in CLASSIFICATION_THRESHOLDS
                    },
                    **{
                        f"Positive_Rate_{cutoff:.2f}": float(np.mean(p >= cutoff))
                        for cutoff in CLASSIFICATION_THRESHOLDS
                    },
                })

            part = validation[["Date", "Ticker", TARGET]].copy()
            part["Algorithm"] = algorithm
            part["Specification"] = specification
            part["Probability"] = val_p
            prediction_parts.append(part)

oof_predictions = pd.concat(prediction_parts, ignore_index=True)
train_validation_stage = pd.DataFrame(fit_rows)

assert oof_predictions["Probability"].between(0, 1).all()
assert not oof_predictions.duplicated(["Date", "Ticker", "Algorithm", "Specification"]).any()

print("Completed validation rebalances:", len(walk_forward_splits))
print("Final evaluation model fits:", len(train_validation_stage) // 2)
print("Validation predictions:", len(oof_predictions))

Tuned XGBoost walk-forward:   0%|          | 0/24 [00:00<?, ?rebalance/s]

Completed validation rebalances: 24
Final evaluation model fits: 48
Validation predictions: 2304


## Training vs validation

In [8]:
keys = ["Rebalance_Date", "Algorithm", "Specification"]

train_vals = train_validation_stage.pivot(index=keys, columns="Stage")
train_vals.columns = [f"{metric}_{stage}" for metric, stage in train_vals.columns]
train_vs_validation = train_vals.reset_index()

train_vs_validation["AUC_Gap"] = (
    train_vs_validation["ROC_AUC_Train"]
    - train_vs_validation["ROC_AUC_Validation"]
)
train_vs_validation["AP_Gap"] = (
    train_vs_validation["Average_Precision_Train"]
    - train_vs_validation["Average_Precision_Validation"]
)

train_validation_summary = (
    train_vs_validation
    .groupby(["Algorithm", "Specification"], as_index=False)[[
        "ROC_AUC_Train",
        "ROC_AUC_Validation",
        "AUC_Gap",
        "Average_Precision_Train",
        "Average_Precision_Validation",
        "AP_Gap",
        "Log_Loss_Train",
        "Log_Loss_Validation",
        "Precision_0.50_Train",
        "Precision_0.50_Validation",
        "Positive_Rate_0.50_Validation",
    ]]
    .mean()
)

display(train_validation_summary.sort_values(
    "ROC_AUC_Validation", ascending=False
).round(4))

,Algorithm,Specification,ROC_AUC_Train,ROC_AUC_Validation,AUC_Gap,Average_Precision_Train,Average_Precision_Validation,AP_Gap,Log_Loss_Train,Log_Loss_Validation,Precision_0.50_Train,Precision_0.50_Validation,Positive_Rate_0.50_Validation
0,XGBoost,With Regimes,0.8295,0.6866,0.1429,0.8751,0.7500,0.1251,0.5093,0.6463,0.7700,0.6638,0.8394
1,XGBoost,Without Regimes,0.8294,0.6844,0.1450,0.8750,0.7397,0.1352,0.5094,0.6476,0.7695,0.6750,0.8385


## Classification metrics and thresholds

In [9]:
direction_rows = []

# One pooled six-year result per model/specification/threshold
for (algorithm, specification), part in oof_predictions.groupby(
    ["Algorithm", "Specification"], sort=True
):
    for threshold in CLASSIFICATION_THRESHOLDS:
        direction_rows.append({
            "Algorithm": algorithm,
            "Specification": specification,
            "Threshold": threshold,
            **direction_metrics(part, threshold),
        })

# Always-positive benchmark, using each asset-date once
baseline_source = oof_predictions.loc[
    (oof_predictions["Algorithm"] == ALGORITHMS[0])
    & (oof_predictions["Specification"] == "Without Regimes")
]

for threshold in CLASSIFICATION_THRESHOLDS:
    direction_rows.append({
        "Algorithm": "AlwaysPositive",
        "Specification": "Baseline",
        "Threshold": threshold,
        **direction_metrics(
            baseline_source.assign(Probability=1.0),
            threshold
        ),
    })

direction_results = pd.DataFrame(direction_rows)

display(
    direction_results[[
        "Algorithm", "Specification", "Threshold",
        "Precision", "Always_Positive_Precision",
        "Precision_Lift_pp", "Recall", "Specificity",
        "Balanced_Accuracy", "Positive_Prediction_Rate",
        "ROC_AUC", "Average_Precision"
    ]]
    .sort_values(["Threshold", "Precision"], ascending=[True, False]).round(4))

,Algorithm,Specification,Threshold,Precision,Always_Positive_Precision,Precision_Lift_pp,Recall,Specificity,Balanced_Accuracy,Positive_Prediction_Rate,ROC_AUC,Average_Precision
0,XGBoost,With Regimes,0.50,0.6722,0.6528,1.9404,0.8644,0.2075,0.5359,0.8394,0.5936,0.7091
5,XGBoost,Without Regimes,0.50,0.6667,0.6528,1.3889,0.8564,0.1950,0.5257,0.8385,0.5819,0.6995
10,AlwaysPositive,Baseline,0.50,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
1,XGBoost,With Regimes,0.55,0.6947,0.6528,4.1934,0.7686,0.3650,0.5668,0.7222,0.5936,0.7091
6,XGBoost,Without Regimes,0.55,0.6795,0.6528,2.6709,0.7753,0.3125,0.5439,0.7448,0.5819,0.6995
11,AlwaysPositive,Baseline,0.55,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
2,XGBoost,With Regimes,0.60,0.7109,0.6528,5.8116,0.6769,0.4825,0.5797,0.6215,0.5936,0.7091
7,XGBoost,Without Regimes,0.60,0.7053,0.6528,5.2529,0.6715,0.4725,0.5720,0.6215,0.5819,0.6995
12,AlwaysPositive,Baseline,0.60,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
3,XGBoost,With Regimes,0.65,0.7277,0.6528,7.4971,0.5545,0.6100,0.5823,0.4974,0.5936,0.7091


## Probability-threshold screening


In [10]:
screening_daily_rows = []

for (algorithm, specification, date), daily in oof_predictions.groupby(
    ["Algorithm", "Specification", "Date"], sort=True
):
    ranked = daily.sort_values(["Probability", "Ticker"], ascending=[False, True], kind="stable")
    baseline = float(daily[TARGET].mean())

    # Select every asset at or above each threshold; no ranking cap.
    for threshold in CLASSIFICATION_THRESHOLDS:
        selected = ranked.loc[ranked["Probability"] >= threshold]
        rule = "Probability Threshold"
        screening_daily_rows.append({
            "Date": date,
            "Algorithm": algorithm,
            "Specification": specification,
            "Rule": rule,
            "Threshold": threshold,
            "Available_Assets": len(daily),
            "Selected_Assets": len(selected),
            "Correct_Positives": int(selected[TARGET].sum()),
            "Same_Date_Baseline": baseline,
        })

screening_daily = pd.DataFrame(screening_daily_rows)


screening_rows = []

for (algorithm, specification, rule, threshold), part in screening_daily.groupby(
    ["Algorithm", "Specification", "Rule", "Threshold"],
    dropna=False,
    sort=True
):
    screening_rows.append({
        "Algorithm": algorithm,
        "Specification": specification,
        "Rule": rule,
        "Threshold": threshold,
        **screening_summary(part),
    })

screening_results = pd.DataFrame(screening_rows)

display(
    screening_results[[
        "Algorithm",
        "Specification",
        "Rule",
        "Threshold",
        "Dates",
        "Selected_Asset_Dates",
        "Precision",
        "Matched_Date_Baseline",
        "Precision_Lift_pp",
        "Mean_Assets",
        "Min_Assets",
        "Max_Assets",
        "Cash_Only_Dates"
    ]]
    .sort_values(["Rule", "Threshold", "Precision_Lift_pp"], ascending=[True, True, False]).round(4))

# Select only among threshold-only rules on validation data, never on the test set.
# Reuse saved hyperparameters when tuning is off, but reselect screening
# so an older saved Hybrid Top-K configuration cannot persist.
selection_candidates = screening_results.loc[
    (screening_results["Mean_Assets"] >= MIN_AVERAGE_SELECTED_ASSETS)
    & screening_results["Precision_Lift_pp"].notna()
].copy()
if selection_candidates.empty:
    raise ValueError(
        f"No probability threshold meets the minimum average coverage of "
        f"{MIN_AVERAGE_SELECTED_ASSETS} assets per validation rebalance. "
        "Review screening_metrics.csv / validation distributions before changing this rule."
    )
selected_strategy = selection_candidates.sort_values(
    ["Precision_Lift_pp", "Mean_Assets"], ascending=[False, False],
).iloc[0]
assert selected_strategy["Rule"] == "Probability Threshold"
print(f"Selected threshold-only strategy on validation, min mean assets: "
      f"{MIN_AVERAGE_SELECTED_ASSETS}")

STRATEGY_RESULT_COLUMNS = [
    "Date", "Algorithm", "Specification", "Rule", "Threshold",
    "Train_Obs", "Available_Assets", "Selected_Assets", "Correct_Positives",
    "Precision", "Baseline", "Precision_Lift_pp",
    "Ticker", "Probability", "Predicted_Sign", "Actual_Sign",
    "Forward_Return_60d", "Outcomes_Available",
]

# Asset-level results for the selected screening strategy
selected_assets = oof_predictions.loc[
    (oof_predictions["Algorithm"] == selected_strategy["Algorithm"])
    & (oof_predictions["Specification"] == selected_strategy["Specification"])
    & (oof_predictions["Probability"] >= selected_strategy["Threshold"])
].copy()

selected_assets = selected_assets.sort_values(["Date", "Probability", "Ticker"], ascending=[True, False, True])

# Every above-threshold asset is kept, without Top-K truncation.
selection_rule = "Probability Threshold"

selected_assets["Predicted_Sign"] = "Positive"
selected_assets["Actual_Sign"] = np.where(
    selected_assets[TARGET] == 1, "Positive", "Negative"
)

# Use the same asset-level output schema for validation and test results.
# Pull the daily totals directly from the selected screening rule, retaining cash-only dates.
selected_daily = screening_daily.loc[
    (screening_daily["Algorithm"] == selected_strategy["Algorithm"])
    & (screening_daily["Specification"] == selected_strategy["Specification"])
    & (screening_daily["Rule"] == selected_strategy["Rule"])
    & np.isclose(
        screening_daily["Threshold"].astype(float),
        float(selected_strategy["Threshold"]),
    )
].copy()
assert len(selected_daily) == len(walk_forward_splits)

# Training observations per rebalance are already recorded in the validation schedule.
train_count_by_date = pd.DataFrame(validation_summary).set_index("Rebalance_Date")["Train_Obs"]
selected_daily["Train_Obs"] = selected_daily["Date"].map(train_count_by_date)
assert selected_daily["Train_Obs"].notna().all()
selected_daily["Precision"] = (
    selected_daily["Correct_Positives"]
    / selected_daily["Selected_Assets"].replace(0, np.nan)
)
selected_daily["Precision_Lift_pp"] = (
    100.0 * (selected_daily["Precision"] - selected_daily["Same_Date_Baseline"])
)
selected_daily["Outcomes_Available"] = True

# Add the observed 60-session return for selected assets only; validation labels are mature.
validation_selected_assets = selected_assets[[
    "Date", "Ticker", "Probability", "Predicted_Sign", "Actual_Sign"
]].merge(
    df[["Date", "Ticker", "Forward_Return_60d"]],
    on=["Date", "Ticker"],
    how="left",
    validate="one_to_one",
)

selected_strategy_details = selected_daily.merge(
    validation_selected_assets,
    on="Date", how="left", validate="one_to_many",
).rename(columns={"Same_Date_Baseline": "Baseline"})
selected_strategy_details = selected_strategy_details[STRATEGY_RESULT_COLUMNS]
assert selected_strategy_details["Outcomes_Available"].all()

print("\nSELECTED SCREENING STRATEGY")
print(f"Algorithm: {selected_strategy['Algorithm']}")
print(f"Specification: {selected_strategy['Specification']}")
print(f"Rule: {selected_strategy['Rule']}")
print(f"Threshold: {selected_strategy['Threshold']:.2f}")
print(f"Precision: {selected_strategy['Precision']:.2%}")
print(f"Precision lift: {selected_strategy['Precision_Lift_pp']:.2f} pp")
print(f"Mean selected assets: {selected_strategy['Mean_Assets']:.2f}")
print(f"Min selected assets: {int(selected_strategy['Min_Assets'])}")
print(f"Max selected assets: {int(selected_strategy['Max_Assets'])}")
print(f"Cash-only dates: {int(selected_strategy['Cash_Only_Dates'])}")

,Algorithm,Specification,Rule,Threshold,Dates,Selected_Asset_Dates,Precision,Matched_Date_Baseline,Precision_Lift_pp,Mean_Assets,Min_Assets,Max_Assets,Cash_Only_Dates
0,XGBoost,With Regimes,Probability Threshold,0.50,24,967,0.6722,0.6518,2.0424,40.2917,0,48,1
5,XGBoost,Without Regimes,Probability Threshold,0.50,24,966,0.6667,0.6495,1.7189,40.2500,4,48,0
6,XGBoost,Without Regimes,Probability Threshold,0.55,24,858,0.6795,0.6449,3.4577,35.7500,0,48,1
1,XGBoost,With Regimes,Probability Threshold,0.55,24,832,0.6947,0.6641,3.0649,34.6667,0,48,1
7,XGBoost,Without Regimes,Probability Threshold,0.60,24,716,0.7053,0.6604,4.4896,29.8333,0,48,2
2,XGBoost,With Regimes,Probability Threshold,0.60,24,716,0.7109,0.6679,4.3005,29.8333,0,48,2
3,XGBoost,With Regimes,Probability Threshold,0.65,24,573,0.7277,0.6700,5.7701,23.8750,0,46,4
8,XGBoost,Without Regimes,Probability Threshold,0.65,24,603,0.7164,0.6645,5.1962,25.1250,0,47,5
4,XGBoost,With Regimes,Probability Threshold,0.70,24,432,0.7454,0.6875,5.7919,18.0000,0,44,9
9,XGBoost,Without Regimes,Probability Threshold,0.70,24,417,0.7266,0.6738,5.2858,17.3750,0,43,8


Selected threshold-only strategy on validation, min mean assets: 20

SELECTED SCREENING STRATEGY
Algorithm: XGBoost
Specification: With Regimes
Rule: Probability Threshold
Threshold: 0.65
Precision: 72.77%
Precision lift: 5.77 pp
Mean selected assets: 23.88
Min selected assets: 0
Max selected assets: 46
Cash-only dates: 4


## Test the selected strategy

In [11]:
# At each test rebalance, refit XGBoost on labels known strictly before that date
# Continue the same 60-session schedule used to anchor validation dates

test_positions = np.arange(anchor_pos, len(trading_dates), REBALANCE_STEP)
test_dates = trading_dates[test_positions]

selected_features = SPECIFICATIONS[str(selected_strategy["Specification"])]
test_rule = str(selected_strategy["Rule"])
test_threshold = float(selected_strategy["Threshold"])

test_selection_parts = []
test_rebalance_rows = []
test_prediction_parts = []

for rebalance_date in tqdm(test_dates, desc="XGBoost test walk-forward", unit="rebalance"):
    # The target for a training observation must be known before this rebalance
    train = df.loc[
        (df["Date"] < rebalance_date)
        & (df["_Target_End_Date"] < rebalance_date)
        & valid_target
    ]
    candidates = df.loc[df["Date"] == rebalance_date].copy()
    if train.empty or candidates.empty:
        continue
    if train[TARGET].nunique() != 2:
        raise ValueError(f"Training targets have only one class at {rebalance_date}.")
    assert train["_Target_End_Date"].max() < rebalance_date

    model = create_model()  # Best model parameters (tuned or loaded)
    model.fit(X(train, selected_features), train[TARGET].to_numpy(dtype=int))

    ranked = candidates[["Date", "Ticker", TARGET, "Forward_Return_60d", "_Target_End_Date"]].copy()
    ranked["Probability"] = model.predict_proba(X(candidates, selected_features))[:, 1]
    ranked = ranked.sort_values(["Probability", "Ticker"], ascending=[False, True], kind="stable")

    eligible = ranked.loc[ranked["Probability"] >= test_threshold]
    selected = eligible.copy()  # All assets with Probability >= selected threshold

    # Outcomes are measured only when all universe labels for this date have matured
    complete_outcomes = bool(
    candidates[TARGET].isin([0, 1]).all()
    and candidates["Forward_Return_60d"].notna().all()
    )

    # Preserve full-universe predictions in memory for test ROC-AUC and average precision.
    all_candidates = ranked[["Date", "Ticker", TARGET, "Probability", "Forward_Return_60d"]].copy()
    all_candidates["Outcomes_Available"] = complete_outcomes
    if not complete_outcomes:
        all_candidates[TARGET] = np.nan
        all_candidates["Forward_Return_60d"] = np.nan
    test_prediction_parts.append(all_candidates)

    n_selected = len(selected)
    n_correct = int(selected[TARGET].sum()) if complete_outcomes else np.nan
    daily_precision = n_correct / n_selected if complete_outcomes and n_selected else np.nan
    baseline = float(candidates[TARGET].mean()) if complete_outcomes else np.nan
    daily_lift = 100.0 * (daily_precision - baseline) if complete_outcomes and n_selected else np.nan

    test_rebalance_rows.append({
        "Date": rebalance_date,
        "Algorithm": "XGBoost",
        "Specification": str(selected_strategy["Specification"]),
        "Rule": test_rule,
        "Threshold": test_threshold,
        "Train_Obs": len(train),
        "Available_Assets": len(candidates),
        "Selected_Assets": n_selected,
        "Correct_Positives": n_correct,
        "Precision": daily_precision,
        "Same_Date_Baseline": baseline,
        "Precision_Lift_pp": daily_lift,
        "Outcomes_Available": complete_outcomes,
    })

    if n_selected:
        selected["Predicted_Sign"] = "Positive"
        selected["Actual_Sign"] = (
            np.where(selected[TARGET] == 1, "Positive", "Negative")
            if complete_outcomes else pd.NA
        )
        if not complete_outcomes:
            selected["Forward_Return_60d"] = np.nan
        selected["Outcomes_Available"] = complete_outcomes
        test_selection_parts.append(selected[[
            "Date", "Ticker", "Probability", "Predicted_Sign",
            "Actual_Sign", "Forward_Return_60d", "Outcomes_Available"
        ]])

if not test_rebalance_rows:
    raise ValueError(
        "No test rebalances were evaluated. Check test dates and the available feature data."
    )

test_rebalance_summary = pd.DataFrame(test_rebalance_rows)
test_asset_columns = [
    "Date", "Ticker", "Probability", "Predicted_Sign", "Actual_Sign",
    "Forward_Return_60d", "Outcomes_Available",
]
test_selected_assets = (
    pd.concat(test_selection_parts, ignore_index=True)
    if test_selection_parts else pd.DataFrame(columns=test_asset_columns)
)

# One output contains dated rebalance totals and selected assets (cash-only dates retained)
selected_strategy_test_results = test_rebalance_summary.drop(
    columns="Outcomes_Available"
).merge(
    test_selected_assets,
    on="Date", how="left", validate="one_to_many",
)
# Cash-only rebalance rows should still disclose outcome maturity
selected_strategy_test_results["Outcomes_Available"] = (
    selected_strategy_test_results["Outcomes_Available"].fillna(
        selected_strategy_test_results["Date"].map(
            test_rebalance_summary.set_index("Date")["Outcomes_Available"]
        )
    ).astype(bool)
)


# Align the test strategy export with the validation asset-level schema.
selected_strategy_test_results = selected_strategy_test_results.rename(
    columns={"Same_Date_Baseline": "Baseline"}
)[STRATEGY_RESULT_COLUMNS]

# Report classification performance only on test dates with fully matured outcomes
scored = test_rebalance_summary.loc[test_rebalance_summary["Outcomes_Available"]].copy()
scored_selected = int(scored["Selected_Assets"].sum())
scored_correct = int(scored["Correct_Positives"].sum()) if not scored.empty else 0
scored_precision = scored_correct / scored_selected if scored_selected else np.nan
matched_baseline = (
    float((scored["Selected_Assets"] * scored["Same_Date_Baseline"]).sum() / scored_selected)
    if scored_selected else np.nan
)
scored_lift = (
    100.0 * (scored_precision - matched_baseline) if scored_selected else np.nan
)

print("\nUNSEEN TEST RESULTS:")
print(f"Scheduled test rebalances: {len(test_dates)}")
print(f"Completed test rebalances: {len(test_rebalance_summary)}")
print(f"Fully matured test rebalances: {len(scored)}")
print(f"Pending test rebalances: {len(test_rebalance_summary) - len(scored)}")
print(f"Selected asset-dates (matured): {scored_selected}")
print(f"Test precision (matured): {scored_precision:.2%}")
print(f"Matched baseline (matured): {matched_baseline:.2%}")
print(f"Test precision lift (matured): {scored_lift:.2f} pp")
print(f"Mean selected assets per evaluated date: {test_rebalance_summary['Selected_Assets'].mean():.2f}")
print(f"Cash-only dates: {int((test_rebalance_summary['Selected_Assets'] == 0).sum())}")
if len(scored) == 0:
    print("No mature test outcomes are yet available; selections are exported without outcome metrics.")

# Full-universe test ROC-AUC and average precision (not screening-selected assets only).
test_all_predictions = pd.concat(test_prediction_parts, ignore_index=True)

# Keep every test probability, not only assets meeting the selected threshold.
# Build this once before test metrics so the shared export step can save it.
all_test_export = test_all_predictions.sort_values(
    ["Date", "Probability", "Ticker"], ascending=[True, False, True]
).copy()
all_test_export["Rank"] = all_test_export.groupby("Date").cumcount() + 1
all_test_export = all_test_export[
    ["Date", "Ticker", "Probability", "Rank", TARGET,
     "Forward_Return_60d", "Outcomes_Available"]
]
test_scored_predictions = test_all_predictions.loc[
    test_all_predictions["Outcomes_Available"]
].copy()

classification_by_date = []
for date, part in test_scored_predictions.groupby("Date"):
    y = part[TARGET].to_numpy(dtype=int)
    p = part["Probability"].to_numpy(dtype=float)
    classification_by_date.append({
        "Date": date,
        "Observations": len(part),
        "Baseline_Precision": float(y.mean()),
        "ROC_AUC": safe_auc(y, p),
        "Average_Precision": safe_ap(y, p),
    })

test_classification_by_rebalance = pd.DataFrame(classification_by_date, columns=[
    "Date", "Observations", "Baseline_Precision", "ROC_AUC", "Average_Precision"
])

if not test_scored_predictions.empty:
    test_y = test_scored_predictions[TARGET].to_numpy(dtype=int)
    test_p = test_scored_predictions["Probability"].to_numpy(dtype=float)
    pooled_auc = safe_auc(test_y, test_p)
    pooled_ap = safe_ap(test_y, test_p)
else:
    pooled_auc = np.nan
    pooled_ap = np.nan

test_classification_metrics = pd.DataFrame([{
    "Algorithm": "XGBoost",
    "Specification": str(selected_strategy["Specification"]),
    "Observations": len(test_scored_predictions),
    "Scored_Dates": test_scored_predictions["Date"].nunique(),
    "Pooled_ROC_AUC": pooled_auc,
    "Pooled_Average_Precision": pooled_ap,
    "Mean_ROC_AUC": test_classification_by_rebalance["ROC_AUC"].mean(),
    "Mean_Average_Precision": test_classification_by_rebalance["Average_Precision"].mean(),
}])

print("\nUNSEEN TEST CLASSIFICATION METRICS")
display(test_classification_metrics.round(4))
print(f"Test mean within-date ROC-AUC: {test_classification_metrics.loc[0, 'Mean_ROC_AUC']:.4f}")
print(f"Test pooled ROC-AUC: {test_classification_metrics.loc[0, 'Pooled_ROC_AUC']:.4f}")

XGBoost test walk-forward:   0%|          | 0/20 [00:00<?, ?rebalance/s]


UNSEEN TEST RESULTS:
Scheduled test rebalances: 20
Completed test rebalances: 20
Fully matured test rebalances: 20
Pending test rebalances: 0
Selected asset-dates (matured): 365
Test precision (matured): 73.97%
Matched baseline (matured): 70.37%
Test precision lift (matured): 3.60 pp
Mean selected assets per evaluated date: 18.25
Cash-only dates: 6

UNSEEN TEST CLASSIFICATION METRICS


,Algorithm,Specification,Observations,Scored_Dates,Pooled_ROC_AUC,Pooled_Average_Precision,Mean_ROC_AUC,Mean_Average_Precision
0,XGBoost,With Regimes,960,20,0.6613,0.7166,0.5889,0.6754


Test mean within-date ROC-AUC: 0.5889
Test pooled ROC-AUC: 0.6613


## Checks

In [12]:
assert oof_predictions["Date"].nunique() == len(walk_forward_splits)
if config.HYPERPARAMETER_TUNING:
    assert len(hyperparameter_search_results) == N_ITER
else:
    assert loaded_config is not None
    assert best_xgb_params == loaded_config["model_params"]
assert oof_predictions[["Algorithm", "Specification"]].drop_duplicates().shape[0] == 2
assert oof_predictions["Probability"].between(0, 1).all()
assert len(direction_results) == 3 * len(CLASSIFICATION_THRESHOLDS)
assert len(screening_results) == 2 * len(CLASSIFICATION_THRESHOLDS)
assert screening_results["Rule"].eq("Probability Threshold").all()
assert selected_strategy["Mean_Assets"] >= MIN_AVERAGE_SELECTED_ASSETS
assert selected_strategy_test_results["Rule"].eq("Probability Threshold").all()
assert test_all_predictions["Probability"].between(0, 1).all()
assert not test_all_predictions.duplicated(["Date", "Ticker"]).any()

assert not test_rebalance_summary["Date"].duplicated().any()
assert test_rebalance_summary["Date"].ge(TEST_START).all()
assert selected_strategy_test_results["Probability"].dropna().between(0, 1).all()
assert selected_strategy_test_results["Rule"].eq(test_rule).all()
assert selected_strategy_test_results["Specification"].eq(str(selected_strategy["Specification"])).all()
print("All validation and test checks passed.")
assert selected_strategy_details.columns.tolist() == STRATEGY_RESULT_COLUMNS
assert selected_strategy_test_results.columns.tolist() == STRATEGY_RESULT_COLUMNS
assert not selected_strategy_details.dropna(subset=["Ticker"]).duplicated(["Date", "Ticker"]).any()
assert not selected_strategy_test_results.dropna(subset=["Ticker"]).duplicated(["Date", "Ticker"]).any()
assert selected_strategy_details["Probability"].dropna().between(0, 1).all()
assert selected_strategy_test_results["Probability"].dropna().between(0, 1).all()
assert test_classification_by_rebalance["Date"].nunique() == test_rebalance_summary["Outcomes_Available"].sum()

All validation and test checks passed.


## Export results

In [13]:
# Export validation and test results to the output folder
csv_files = {
    "validation_classification_metrics.csv": direction_results,
    "train_validation_metrics.csv": train_validation_summary,
    "train_validation_by_rebalance.csv": train_vs_validation,
    "screening_metrics.csv": screening_results,
    "selected_strategy_validation_results.csv": selected_strategy_details,
    "selected_strategy_test_results.csv": selected_strategy_test_results,
    "test_classification_metrics.csv": test_classification_metrics,
    "test_classification_by_rebalance.csv": test_classification_by_rebalance,
    "all_test_predictions.csv": all_test_export,
}

if config.HYPERPARAMETER_TUNING:
    csv_files.update({
        "hyperparameter_search.csv": hyperparameter_search_results,
        "best_hyperparameters.csv": best_hyperparameters,
    })

for filename, table in csv_files.items():
    table.to_csv(OUTPUT / filename, index=False)

# Save the validation-selected threshold even when model tuning is disabled.
model_params = create_model(params=best_xgb_params).get_params()
selected_config = {
    "algorithm": str(selected_strategy["Algorithm"]),
    "specification": str(selected_strategy["Specification"]),
    "model_params": {
        **best_xgb_params,
        "reg_alpha": model_params["reg_alpha"],
        "tree_method": model_params["tree_method"],
        "objective": model_params["objective"],
        "eval_metric": model_params["eval_metric"],
        "n_jobs": model_params["n_jobs"],
        "random_state": model_params["random_state"],
    },
    "screening": {
        "rule": "Probability Threshold",
        "threshold": float(selected_strategy["Threshold"]),
        "top_k": None,
        "min_average_assets": MIN_AVERAGE_SELECTED_ASSETS,
        "threshold_candidates": list(CLASSIFICATION_THRESHOLDS),
    },
}
with selected_config_path.open("w", encoding="utf-8") as f:
    json.dump(
        selected_config, f, indent=4,
        default=lambda x: x.item() if isinstance(x, np.generic) else str(x),
    )
print(f"Screening metrics, test predictions and selected configuration saved to {OUTPUT}/")


Screening metrics, test predictions and selected configuration saved to classification_results\xgboost/
